# 13 - Train a Recurrent-Pass DQN Model Offline

Same offline loop as `02_train_offline_dqn.ipynb`, with **recurrent-depth** refinement instead of Coconut-style latent thoughts (`12`). Recurrence is a `Model` section, `Recurrence(hidden_dim, num_passes)`, so it is saved with the checkpoint and applied identically at train and inference time:

1. `out = model(inputs)` encodes once and runs the backbone `NUM_PASSES` times. Pass `k` reads `encodings + proj(RMSNorm(pass_{k-1}.last_hidden_state))`: the previous residual stream is normalized before it re-enters the backbone and the original encodings are re-injected every pass. `proj` starts at zero, so at construction every pass equals a plain forward and the optimizer turns the recurrence on.
2. `out.passes` holds every pass's `last_hidden_state` and `predictions`; `out.predictions` is the final pass (what `get_action` uses).
3. `run_train` pairs every online pass with the delayed model's matching pass and averages the `DqnObjective` TD losses, so every pass is supervised at the same loss scale as `02`.
4. No extra thought tokens, no burst sampling, no `LatentReasoner` (a model has one or the other).

Each optimizer step costs roughly `NUM_PASSES` times a plain forward. Cached decode (`09_inference.ipynb`, online rollouts) keeps one KV session per pass, so the evaluated model is the trained one.

Why the adapter: the backbone output goes through the pretrained final RMSNorm, whose learned per-dim gain puts it on a very different scale from the encoder embeddings (on Qwen3-0.6B the gain carries outlier dims). Fed straight back as the next input those outliers compound pass over pass, and in bf16 the later passes' layer contributions round away. The adapter re-normalizes the recycled state without a gain before re-injecting the encodings.

This is a short usage example, not a full experiment.


In [ ]:
import torch

from mouse_core import AdamW
from mouse_core.data import (
    DataLoader,
    Augmenter,
    Tokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective
from mouse_core.models import Model, Polyak, Recurrence, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import NumericEmbedder
from mouse_core.models.heads import DiscreteActionValueHead


DATASET_ID = "mouse-example-dataset"          # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-recurrent"    # Hugging Face model repo for push_model_to_hub
MAX_ACTIONS = 4                               # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                         # vocabulary size for discrete observations
SEQUENCE_LENGTH = 512                         # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                                # sequences per optimizer step
NUM_CYCLES = 2                               # outer train cycles (print cadence)
TRAIN_STEPS = 50                             # optimizer updates per cycle (passed to run_train)
POLYAK_TAU_HEADS = 0.0001                     # delayed Q-head interpolation (0 = frozen, 1 = copy of the online heads)
POLYAK_TAU_ENCODER = 0.01                     # delayed encoder interpolation
POLYAK_TAU_BACKBONE = 0.01                    # delayed backbone interpolation
NUM_PASSES = 4                                # backbone passes per forward (Recurrence section; saved with the model)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split='train', force_download=True)

## Data pipeline

`DataLoader` samples contiguous windows up to `sequence_length` (a max) from one or more datastores. Each sequence may be shorter than the max depending on where the window starts in the store.

Pipeline order: `augmenter → tokenizer → pack → embedder`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` for shared draws within a `reseed` generation). Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Tokenizer** | `dict → StepTokens` (`input_field` / `output_field`; `objective_fields=` is `action` / `reward` / `episode_done` / `task_done`; `grouping_field=`) |

Compose `train_transform = compose(augmenter, tokenizer)`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
The token layout matches `02` — recurrent passes refine the same token stream, so no extra prompt token is required.


In [ ]:
# Pipeline order: augmenter → tokenizer

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

tokenizer = Tokenizer(
    input_fields=[
        {
            "type": "discrete",
            "input_field": "action",
        },
        {
            "type": "discrete",
            "input_field": "observation",
        },
        {
            "type": "fourier",
            "input_field": "reward",
        },
        {
            "type": "discrete",
            "input_field": "episode_done",
        },
        {
            "type": "learnable",
            "output_field": "value",
            "tokens": 1,
            "head_output": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "episode_done",
        },
        {
            "input_field": "task_done",
        },
    ],
    grouping_field="task_index",
)

train_transform = compose(augmenter, tokenizer)

loader = DataLoader(
    stores=stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)



## Build The Model

A Mouse Core `Model` has the same three pieces as `02` plus a `Recurrence` section:

- `NumericEmbedder` maps a tokenized `TokenBatch` (modalities keyed by name; add `vocab_size` / `std` here; `fourier` / `continuous` also need `fourier_min` / `fourier_max`) into vectors. It runs once per forward.
- `Qwen3Backbone` processes those tokens with a transformer backbone, `NUM_PASSES` times. Pass `k` reads `recurrence(encodings, pass_{k-1}.last_hidden_state)`.
- `Recurrence(hidden_dim, num_passes)` is the adapter between passes: `encodings + proj(RMSNorm(last_hidden_state))`, `proj` zero-initialised.
- `DiscreteActionValueHead` predicts one value per discrete action after **each** pass.

The backbone exposes `hidden_dim`, and the embedder and head use that same value so the pieces connect cleanly.

`NumericEmbedder` modality types used here:

- `discrete` for integer IDs such as actions, observations, and episode/task done codes.
- `fourier` for scalar numeric values such as rewards.
- `learnable` for the trailing `value` token (no step field; flagged `head_output: True` so Q / action outputs are read from it).

`Model(...)` wraps the pieces behind a single forward call. `recurrence=` cannot be combined with a `LatentReasoner` on the same model.


In [ ]:
backbone = Qwen3Backbone(
    train_kernel="flex",
    decode_kernel="flex",
    dtype=torch.float32,
    pretrained="Qwen/Qwen3-0.6B",
)

encoder = NumericEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "type": "discrete",
            "field": "action",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "discrete",
            "field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "fourier",
            "field": "reward",
            "std": 0.02,
            "positions": 1,
            "fourier_min": 0.01,
            "fourier_max": 10.0,
        },
        {
            "type": "discrete",
            "field": "episode_done",
            "vocab_size": 3,
            "std": 0.02,
            "positions": 1,
        },
        {
            "type": "learnable",
            "field": "value",
            "tokens": 1,
            "std": 0.02,
            "positions": 1,
        },
    ],
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

recurrence = Recurrence(hidden_dim=backbone.hidden_dim, num_passes=NUM_PASSES)

model = Model(
    encoder=encoder,
    backbone=backbone,
    heads=head,
    action_head="action_value",
    reasoner=None,
    recurrence=recurrence,
).train().to(device)
print(model)



## Training Phase

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`:

1. `inputs, objective_data = loader.next_batch()` samples ragged step windows (up to `SEQUENCE_LENGTH`).
2. `out = model(inputs)` runs encoder once, backbone `NUM_PASSES` times, heads after each pass. `out.passes[k]` has that pass's `last_hidden_state` and `predictions`.
3. `delayed_model(inputs)` runs the same `NUM_PASSES` passes through the delayed model under `torch.no_grad()`; `objective(...)` on pass `k` of both models gives that pass's TD loss. The losses are **averaged**, so the update has the same scale as `02` regardless of `NUM_PASSES`. Metrics come from the last pass.
4. `AdamW` updates weights. The backbone, encoder, and heads are fp32 (`dtype=torch.float32`), so every update lands in fp32 with no master weights.
5. `delayed_model = model.delayed_copy()` is a frozen copy of the online model — the fp32 encoder, backbone, recurrence adapter, and Q head are copied. `tau_backbone` also interpolates the recurrence adapter. `polyak.update(tau_heads=POLYAK_TAU_HEADS, tau_encoder=POLYAK_TAU_ENCODER, tau_backbone=POLYAK_TAU_BACKBONE)` interpolates each section toward the online model after the optimizer step: `0` keeps it frozen, `1` copies the online weights (no delay). Every interpolated parameter is fp32, so a small `tau` is never rounded away.

`DqnObjective` interprets `episode_done` and `task_done` (each `0`/`1`/`2`) through separate discount factors. The bootstrap is multiplied by the episode gamma, then by the task gamma (`1.0` when `task_done` is `0`). When a task ends both fire, so a task gamma of `0.0` zeros the whole term.


In [ ]:
optimizer = AdamW(
    model.parameters(),
    lr=1e-05,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)
delayed_model = model.delayed_copy()
polyak = Polyak(model, delayed_model)
objective = DqnObjective(
    gamma_step=1.0,
    gamma_episode_terminal=1.0,
    gamma_episode_truncated=1.0,
    gamma_task_terminal=0.0,
    gamma_task_truncated=0.0,
    grouping_field="task_index",
)

def run_train(*, model: Model, delayed_model: Model, polyak: Polyak, optimizer: AdamW, objective: DqnObjective, loader: DataLoader, num_steps: int) -> tuple[torch.Tensor, dict[str, float]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        data = objective_data.to(device)
        out = model(inputs)  # NUM_PASSES backbone passes; out.passes has each one
        with torch.no_grad():
            delayed_out = delayed_model(inputs)
        pass_losses = []
        for current, delayed_pass in zip(out.passes, delayed_out.passes):
            step_loss, metrics = objective(
                data,
                current.predictions,
                delayed_pass.predictions,
            )
            pass_losses.append(step_loss)
        loss = torch.stack(pass_losses).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        polyak.update(
            tau_heads=POLYAK_TAU_HEADS,
            tau_encoder=POLYAK_TAU_ENCODER,
            tau_backbone=POLYAK_TAU_BACKBONE,
        )
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(
        model=model,
        delayed_model=delayed_model,
        polyak=polyak,
        optimizer=optimizer,
        objective=objective,
        loader=loader,
        num_steps=TRAIN_STEPS,
    )
    print(f"cycle={cycle} train  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}")
loader.close()


## Push To The Hub

`push_model_to_hub` saves the model architecture and weights together. Later, `load_model` can reconstruct the full `Model` without repeating the embedder, backbone, and head definitions. The `Recurrence` section (`num_passes` and adapter weights) is part of the checkpoint, so `09_inference.ipynb` runs the same `NUM_PASSES` recurrence with cached decode — one KV session per pass.


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model=model, repo_id=MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")